In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from analyses.rust import contact
from analyses import io
from pathlib import Path

In [ ]:
latpaths = {}
for replica in Path("../runs/cil_med-act/energy-10/").iterdir():
    rep = int(replica.name)
    latpaths[rep] = []
    for file in replica.joinpath("lattices").iterdir():
        latpaths[rep].append((
            file,
            str(file).replace("lattices", "act_lattices")
        ))

In [ ]:
def acts_to_df(acts):
    return pl.DataFrame({
        "label": ["c"] * len(acts.cell) + ["m"] * len(acts.medium) + ["s"] * len(acts.solid),
        "act": acts.cell + acts.medium + acts.solid,
    })

In [ ]:
klats = []
dfs = []
clat_path, alat_path = latpaths[0][100]
clat, alat = pl.read_parquet(clat_path), pl.read_parquet(alat_path)
for r in range(0, 17):
    res = contact.kernel_act(clat, alat, r, True)
    klat = res[1]
    klat = np.array(klat).reshape(500, 500)
    klats.append(klat)
    dfs.append(acts_to_df(res[0]).with_columns(radius=r))
klat_3d = np.array(klats)
rdf = pl.concat(dfs)

In [ ]:
fig = px.imshow(np.where(klat_3d == 0, np.where(clat.to_numpy().T == "m", 0, 2), 1), animation_frame=0, width=800, height=800)
fig.update_layout(
    template="plotly_white",
    xaxis_range=[100, 400],
    yaxis_range=[100, 400], 
    yaxis_autorange=False,
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    xaxis_constrain="domain",
    xaxis_visible=False,
    yaxis_visible=False,
    margin=dict(l=0, r=0, t=0, b=0),
)
# io.save_plot(fig, "../plots/act_kernel_img")

In [ ]:
px.violin(rdf, y="act", color="label", animation_frame="radius")

In [ ]:
cdf = rdf.group_by(["radius", "label"]).agg(
    mean=pl.col("act").mean()
).sort(
    "radius"
).pivot(
    on="label",
    index="radius", 
    values="mean"
).with_columns(
    diff=pl.col("m") - pl.col("c"),
    mdiff=pl.col("m") / pl.col("c")
).unpivot(
    index="radius"
)
fig = px.bar(cdf.filter(pl.col("radius") < 5), x="radius", y="value", color="variable", barmode="group")
fig.update_layout(
    template="plotly_white",
    width=600,
    height=400,
    xaxis_title="kernel radius",
    yaxis_title="mean act"
)
io.save_plot(fig, "../plots/act_kernel")